In [ ]:
# product-intelligence-system/
# │
# ├── data/
# │   ├── raw/                    # Original datasets (Kaggle, external sources)
# │   │   ├── amazon_reviews.csv
# │   │   └── electronics_products.csv
# │   │ 
# │   └── processed/              # Cleaned + feature-engineered datasets
# │       └── processed_reviews.csv
# │
# ├── notebooks/
# │   └── full_notebook.ipynb        # End-to-end explained pipeline (recruiter-facing narrative)
# │
# ├── src/
# │   │
# │   ├── __init__.py             # Initializes src as a Python package and exposes core pipeline and intelligence modules for clean imports
# │   │
# │   ├── utils.py                # Logging, timers, IO helpers, normalization, persistence
# │   │
# │   ├── data_loader.py          # Schema standardisation + ID generation + timestamps
# │   ├── preprocessing.py        # Text cleaning + linguistic + statistical feature extraction
# │   ├── features.py             # Sentiment + embeddings + full feature matrix construction
# │   ├── models.py               # Isolation Forest + KMeans + risk signal generation
# │   ├── calibration.py          # Risk features → calibrated fraud probability model
# │   ├── explainability.py       # Human-readable explanations (review + cluster reasoning)
# │   ├── temporal_analysis.py    # Time-based fraud detection + sentiment drift + spikes
# │   ├── evaluation.py           # Full ML evaluation suite (anomaly, clustering, product health)
# │   ├── visualization.py        # Creating key charts for visualization
# │   │ 
# │   └── pipeline.py             # End-to-end orchestration (system entry point)
# │
# ├── run_pipeline.py             # One-command execution entry point
# │
# ├── models/
# │   ├── isolation_forest.pkl    # Trained anomaly detection model
# │   ├── kmeans.pkl              # Clustering model
# │   ├── calibration_model.pkl   # Risk → probability model (logistic regression)
# │   ├── scaler.pkl              # General structured feature scaler
# │   ├── scaler_structured.pkl   # Dedicated structured feature normalization scaler
# │   ├── scaler_embeddings.pkl   # Embedding scaler
# │   └── scaler_calibration.pkl  # Calibration scaler
# │
# ├── outputs/
# │   ├── charts/                 # Saved visualisations (risk trends, clusters)
# │   │   ├── risk_distribution.png     # Distribution of computed risk scores
# │   │   └── cluster_distribution.png  # Cluster frequency distribution
# │   │ 
# │   └── reports/                     # Model outputs, evaluation, and explainability artifacts
# │       ├── evaluation.json               # Full ML evaluation metrics (anomaly detection, clustering, product health diagnostics)
# │       ├── temporal_analysis.json        # Time-based intelligence (drift, spikes, review velocity trends)
# │       ├── cluster_explanations.json     # Cluster-level interpretability (top words, risk patterns, behavioral summaries)
# │       ├── review_explanations.csv      # Row-level explainability for high-risk reviews with human-readable reasoning
# │       └── feature_columns.json         # Feature schema used in model training for reproducibility and debugging
# │
# ├── app/
# │   └── dashboard.py            # Streamlit dashboard (product intelligence + fraud explorer)
# │
# ├── config.yaml                 # Central configuration (paths, hyperparameters, thresholds)
# ├── requirements.txt            # Dependencies (ML + NLP + dashboard stack)
# ├── .gitignore                  # Git exclusions (data, models, outputs, cache)
# └── README.md                  # Project documentation (architecture + results + insights)

In [ ]:
# ┌──────────────────────────────────────────────────────────────┐
# │                  PRODUCT INTELLIGENCE SYSTEM                  │
# │        Multimodal Review Mining & Fraud Detection            │
# └──────────────────────────────────────────────────────────────┘

#                         ┌────────────────────────────┐
#                         │       Raw Data Sources     │
#                         │                            │
#                         │  • Amazon Reviews          │
#                         │  • Product Metadata        │
#                         │  • (Future) Image Data     │
#                         └────────────┬───────────────┘
#                                      │
#                                      ▼
# ───────────────────────────────────────────────────────────────
#                    Data Ingestion Layer
#                  (src/data_loader.py)
# ───────────────────────────────────────────────────────────────
#   • Schema standardisation
#   • ID generation (review_id, product_id)
#   • Timestamp parsing & validation
#                                      │
#                                      ▼
# ───────────────────────────────────────────────────────────────
#                    Preprocessing Layer
#                  (src/preprocessing.py)
# ───────────────────────────────────────────────────────────────
#   • Text cleaning & normalization
#   • Duplicate / short review filtering
#   • Optional language filtering
#   • Structural signals (length, caps, punctuation, z-scores)
#                                      │
#                                      ▼
# ───────────────────────────────────────────────────────────────
#           Feature Engineering & Representation Layer
#                    (src/features.py)
# ───────────────────────────────────────────────────────────────
#   • Sentence-BERT embeddings (semantic representation)
#   • VADER sentiment scoring
#   • Behavioural + linguistic features
#   • Derived signals (mismatch scores, density, z-scores)
#   • Final feature matrix construction (X)
#                                      │
#                                      ▼
# ───────────────────────────────────────────────────────────────
#                       Modeling Layer
#                     (src/models.py)
# ───────────────────────────────────────────────────────────────
#   • Isolation Forest → anomaly detection
#   • KMeans → semantic clustering
#   • Cluster assignment + distance-to-centroid
#   • Feature scaling (structured + embeddings)
#   • Risk feature matrix construction
#                                      │
#                                      ▼
# ───────────────────────────────────────────────────────────────
#                   Risk Intelligence Layer
#          (src/models.py + src/calibration.py)
# ───────────────────────────────────────────────────────────────
#   • Heuristic risk score (multi-signal fusion)
#   • Risk normalization (bounded [0,1])
#   • Logistic regression calibration model
#   • Calibrated fraud probability output
#                                      │
#                                      ▼
# ───────────────────────────────────────────────────────────────
#                 Temporal Intelligence Layer
#               (src/temporal_analysis.py)
# ───────────────────────────────────────────────────────────────
#   • Review velocity (volume trends)
#   • Sentiment drift detection
#   • Fraud spike detection (z-score anomalies)
#   • Multi-window aggregation (1D / 7D / 30D)
#                                      │
#                                      ▼
# ───────────────────────────────────────────────────────────────
#                   Explainability Engine
#               (src/explainability.py)
# ───────────────────────────────────────────────────────────────
#   • Review-level explanations (why flagged)
#   • Risk factor attribution
#   • Cluster-level summaries (top words + behaviour)
#   • Audit-ready interpretability outputs
#                                      │
#                                      ▼
# ───────────────────────────────────────────────────────────────
#                     Evaluation Layer
#                 (src/evaluation.py)
# ───────────────────────────────────────────────────────────────
#   • Anomaly detection metrics (ROC-AUC, PR-AUC)
#   • Clustering quality (silhouette, entropy)
#   • Distribution diagnostics & correlations
#   • Product-level scoring & validation
#                                      │
#                                      ▼
# ───────────────────────────────────────────────────────────────
#                   Visualization Layer
#               (src/visualization.py)
# ───────────────────────────────────────────────────────────────
#   • Risk score distributions
#   • Cluster distributions
#   • Static chart generation → outputs/charts/
#                                      │
#                                      ▼
# ───────────────────────────────────────────────────────────────
#               Output & Persistence Layer
# ───────────────────────────────────────────────────────────────
#   • Processed dataset → data/processed/processed_reviews.csv
#   • Models → models/
#       - isolation_forest.pkl
#       - kmeans.pkl
#       - calibration_model.pkl
#   • Scalers → models/
#       - scaler.pkl
#       - scaler_structured.pkl
#       - scaler_embeddings.pkl
#       - scaler_calibration.pkl
#   • Reports → outputs/reports/
#       - evaluation.json
#       - temporal_analysis.json
#       - cluster_explanations.json
#       - review_explanations.csv
#       - feature_columns.json
#   • Charts → outputs/charts/
#                                      │
#                                      ▼
# ───────────────────────────────────────────────────────────────
#                 Interface & Delivery Layer
# ───────────────────────────────────────────────────────────────
#   • run_pipeline.py → config-driven execution entry point
#   • notebooks/full_notebook.ipynb → end-to-end narrative
#   • app/dashboard.py → Streamlit dashboard
#                                      │
#                                      ▼
# ───────────────────────────────────────────────────────────────
#                Product Intelligence Outputs
# ───────────────────────────────────────────────────────────────
#   • Fraud probability per review
#   • Risk scores + anomaly flags
#   • Product health indicators
#   • Cluster-based behavioural segmentation
#   • Temporal fraud insights (spikes & drift)
#   • Fully explainable AI decisions

In [3]:
import sys

# Core libraries
!{sys.executable} -m pip install pandas numpy scikit-learn scipy

# NLP
!{sys.executable} -m pip install vaderSentiment textblob nltk

# Embeddings / Deep learning
!{sys.executable} -m pip install sentence-transformers torch

# Visualization + Dashboard
!{sys.executable} -m pip install matplotlib seaborn wordcloud plotly streamlit

# Utilities
!{sys.executable} -m pip install tqdm joblib pyyaml

# Notebook support
!{sys.executable} -m pip install jupyter

In [4]:
# Set working directory

import os
os.chdir(r"C:\Users\JoshuaMcCourt\Documents\GitHub Portfolio\Multimodal Product Intelligence + Review Mining System")
print(os.getcwd())

C:\Users\JoshuaMcCourt\Documents\GitHub Portfolio\Multimodal Product Intelligence + Review Mining System


In [5]:
# src/__init__.py

"""
Product Intelligence System

Production-grade NLP + ML pipeline for:

- Review sentiment + behavioural analysis
- Fake review detection (anomaly detection + probabilistic calibration)
- Product / topic clustering
- Explainable AI (review-level + cluster-level interpretability)
- Temporal anomaly detection (spikes, drift, burst patterns)
- Product-level risk scoring system


Architecture Overview

Core Pipeline:

    data_loader → preprocessing → features → models

Intelligence Layers:

    calibration → explainability → temporal_analysis

Evaluation Layer:

    evaluation (diagnostics, metrics, system validation)


Canonical Flow

    df = load_data(...)

    df = preprocess_text_column(df)

    feature_df = build_features(df)

    model_outputs = run_models(feature_df)

    calibrated_outputs = run_calibration(model_outputs)

    review_explanations = generate_review_explanations(calibrated_outputs)

    cluster_explanations = generate_cluster_explanations(calibrated_outputs)

    temporal_results = run_temporal_analysis(calibrated_outputs)

    evaluation_results = run_evaluation(calibrated_outputs)


Data Contracts

run_models(feature_df) must return a DataFrame containing:

    - anomaly_score        : float
    - cluster_label        : int
    - risk_features        : engineered signals derived from model outputs
    - sentiment_score      : float (if used downstream)
    - embedding features   : optional (for clustering / similarity)

run_calibration(model_outputs) expects:

    - anomaly_score
    - cluster_label
    - risk_features

and outputs:

    - fraud_probability    : calibrated probability (0-1)
    - final risk score     : optional composite metric


All downstream modules (explainability, temporal, evaluation)
assume these fields exist.


Design Principles

- Modular but pipeline-driven (clear stage boundaries)
- Deterministic preprocessing, probabilistic modeling
- Separation of:
    preprocessing vs feature engineering vs modeling
- Explainability as a first-class component
- Temporal dynamics treated as independent signal layer
- Reproducible outputs via config-driven execution


Notes

- run_pipeline is intentionally not exposed here
  to avoid circular imports and unintended execution
  in notebook / interactive environments.

- This module exposes the PUBLIC API of the system.
  Only stable, high-level functions are included.
"""


# Core Data Layer

from src.data_loader import load_data
from src.preprocessing import preprocess_text_column


# Feature Engineering

from src.features import build_features


# Modeling Layer

# NOTE:
# run_models performs:
# - Anomaly detection (e.g., Isolation Forest)
# - Clustering (e.g., KMeans)
# - Risk signal construction

from src.models import run_models


# Intelligence Layers

# Calibration: converts raw risk signals → calibrated probability
from src.calibration import run_calibration


# Explainability: human-interpretable outputs
from src.explainability import (
    generate_review_explanations,
    generate_cluster_explanations
)


# Temporal Analysis: time-based anomaly + drift detection
from src.temporal_analysis import run_temporal_analysis


# Evaluation Layer

from src.evaluation import run_evaluation


# Utilities

from src.utils import setup_logger, Timer


# Public API

__all__ = [
    # Core Pipeline
    "load_data",
    "preprocess_text_column",
    "build_features",
    "run_models",

    # Intelligence Layers
    "run_calibration",
    "generate_review_explanations",
    "generate_cluster_explanations",
    "run_temporal_analysis",

    # Evaluation
    "run_evaluation",

    # Utilities
    "setup_logger",
    "Timer",
]

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [6]:
# src/utils.py

import os
import json
import logging
import pandas as pd
from datetime import datetime


# Logging Setup

def setup_logger(name: str = "product_intelligence") -> logging.Logger:
    logger = logging.getLogger(name)

    if not logger.handlers:
        logger.setLevel(logging.INFO)

        handler = logging.StreamHandler()
        formatter = logging.Formatter(
            "[%(asctime)s] [%(name)s] %(levelname)s - %(message)s"
        )
        handler.setFormatter(formatter)
        logger.addHandler(handler)

    return logger


# File Saving Utilities

def _safe_mkdir(path: str):
    dir_path = os.path.dirname(path)
    if dir_path:
        os.makedirs(dir_path, exist_ok=True)


def _append_timestamp(path: str) -> str:
    base, ext = os.path.splitext(path)
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    return f"{base}_{timestamp}{ext}"


def save_dataframe(df: pd.DataFrame, path: str, timestamp: bool = False):
    if timestamp:
        path = _append_timestamp(path)

    _safe_mkdir(path)
    df.to_csv(path, index=False, encoding="utf-8")


def save_json(obj: dict, path: str, timestamp: bool = False):
    if timestamp:
        path = _append_timestamp(path)

    _safe_mkdir(path)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=4, default=str)


def load_json(path: str):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


# Model Persistence

def save_model(model, path: str):
    import joblib
    _safe_mkdir(path)
    joblib.dump(model, path)


def load_model(path: str):
    import joblib
    return joblib.load(path)


# Feature Utilities

def normalize_series(series: pd.Series) -> pd.Series:
    min_val = series.min()
    max_val = series.max()
    denom = max_val - min_val

    if denom == 0:
        return pd.Series(0.0, index=series.index)

    return (series - min_val) / (denom + 1e-9)


def zscore(series: pd.Series) -> pd.Series:
    mean = series.mean()
    std = series.std()

    if std == 0:
        return pd.Series(0.0, index=series.index)

    return (series - mean) / (std + 1e-9)


# Timing / Profiling

class Timer:
    def __init__(self, name="block", logger=None):
        self.name = name
        self.logger = logger

    def __enter__(self):
        self.start = datetime.now()
        return self

    def __exit__(self, *args):
        end = datetime.now()
        duration = (end - self.start).total_seconds()

        message = f"{self.name} took {duration:.4f} seconds"

        if self.logger:
            self.logger.info(message)
        else:
            print(message)

In [7]:
# src/data_loader.py

import pandas as pd
import numpy as np
from typing import Tuple, Optional


# Schema Contract

REQUIRED_COLS = [
    "review_id",
    "product_id",
    "user_id",
    "review_text",
    "rating",
    "timestamp"
]


def validate_schema(df: pd.DataFrame):
    """
    Final safety check after full ingestion pipeline.
    Ensures downstream modules never receive malformed data.
    """

    missing = [c for c in REQUIRED_COLS if c not in df.columns]

    if missing:
        raise ValueError(f"Schema validation failed. Missing columns: {missing}")


# Schema Standardisation

def _standardise_schema(df: pd.DataFrame) -> pd.DataFrame:

    column_map = {
        # Amazon
        "reviews.text": "review_text",
        "reviews.rating": "rating",
        "reviews.title": "summary",

        # Kaggle variants
        "Text": "review_text",
        "Score": "rating",
        "Summary": "summary",

        # Alternative naming
        "reviewText": "review_text",
        "overall": "rating",
        "stars": "rating"
    }

    df = df.copy()

    for src, dst in column_map.items():
        if src in df.columns:
            df = df.rename(columns={src: dst})

    return df


# Timestamp Handling

def _parse_timestamp(df: pd.DataFrame) -> pd.DataFrame:

    df = df.copy()

    timestamp_sources = ["timestamp", "reviewTime", "Time"]

    parsed = False

    for col in timestamp_sources:
        if col in df.columns:
            df["timestamp"] = pd.to_datetime(df[col], errors="coerce")
            parsed = True
            break

    if not parsed:
        df["timestamp"] = pd.date_range(
            start="2020-01-01",
            periods=len(df),
            freq="h"
        )

    # No invalid timestamps
    df = df.dropna(subset=["timestamp"])

    return df


# Cleaning Pipeline

def _clean_data(df: pd.DataFrame) -> pd.DataFrame:

    df = df.copy()

    # Text normalization
    df["review_text"] = df["review_text"].astype(str).str.strip()

    # Numeric safety
    df["rating"] = pd.to_numeric(df["rating"], errors="coerce")

    # Remove invalid rows early
    df = df.dropna(subset=["review_text", "rating"])

    # Deduplication (text-level assumption)
    df = df.drop_duplicates(subset=["review_text"])

    # Length filter (noise reduction)
    df = df[df["review_text"].str.len() > 5]

    # Enforce rating bounds
    df["rating"] = df["rating"].clip(1, 5)

    return df


# ID Generation (Deterministic)

def _generate_ids(df: pd.DataFrame) -> pd.DataFrame:

    df = df.copy()
    n = len(df)

    if "review_id" not in df.columns:
        df["review_id"] = np.arange(n)

    if "product_id" not in df.columns:
        df["product_id"] = [
            f"product_{i % 100}" for i in range(n)
        ]

    if "user_id" not in df.columns:
        df["user_id"] = [
            f"user_{i % 500}" for i in range(n)
        ]

    return df


# Main Review Loader

def load_reviews(path: str, verbose: bool = True) -> pd.DataFrame:

    df = pd.read_csv(path)

    if verbose:
        print("Raw columns:", list(df.columns))

    # Schema alignment
    df = _standardise_schema(df)

    # Hard requirement before proceeding
    if "review_text" not in df.columns:
        raise ValueError("Missing required column: review_text after schema standardisation")

    # Safe default ONLY at ingestion boundary
    if "rating" not in df.columns:
        df["rating"] = 3.0

    # Cleaning
    df = _clean_data(df)

    # Timestamps
    df = _parse_timestamp(df)

    # IDs
    df = _generate_ids(df)

    # Final contract enforcement
    validate_schema(df)

    if verbose:
        print("Final columns:", list(df.columns))
        print(f"Final shape: {df.shape}")

    return df


# Metadata Loader

def load_metadata(path: str, verbose: bool = True) -> Optional[pd.DataFrame]:

    try:
        df_meta = pd.read_csv(path)

        if verbose:
            print("Metadata columns:", list(df_meta.columns))

        return df_meta

    except FileNotFoundError:
        if verbose:
            print("Metadata not found — continuing without it.")
        return None


# Public Entry Point

def load_data(
    reviews_path: str,
    metadata_path: Optional[str] = None,
    verbose: bool = True
) -> Tuple[pd.DataFrame, Optional[pd.DataFrame]]:

    df_reviews = load_reviews(reviews_path, verbose)
    df_meta = load_metadata(metadata_path, verbose) if metadata_path else None

    return df_reviews, df_meta

In [8]:
# src/preprocessing.py

import re
import pandas as pd
import numpy as np
from typing import Optional

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

from src.utils import zscore


# Initialization

try:
    stop_words = set(stopwords.words("english"))
except LookupError:
    # Fallback for environments without nltk data downloaded
    stop_words = set()

lemmatizer = WordNetLemmatizer()


# Precompiled regex (performance improvement)
URL_PATTERN = re.compile(r"http\S+")
HTML_PATTERN = re.compile(r"<.*?>")
NON_ALPHANUM_PATTERN = re.compile(r"[^a-z0-9\s]")
PUNCT_PATTERN = re.compile(r"[!?.,]")


# Language Detection

def is_english(text: str) -> bool:
    text = str(text)

    if not text or len(text) == 0:
        return False

    alpha_chars = re.findall(r"[a-zA-Z]", text)
    alpha_ratio = len(alpha_chars) / max(len(text), 1)

    return alpha_ratio > 0.7


# Text Cleaning

def clean_text(text: str) -> str:

    text = str(text).lower().strip()

    # Fast regex cleaning
    text = URL_PATTERN.sub("", text)
    text = HTML_PATTERN.sub("", text)
    text = NON_ALPHANUM_PATTERN.sub(" ", text)

    words = text.split()

    cleaned_words = []
    for w in words:
        if w in stop_words:
            continue
        if len(w) <= 1:
            continue

        cleaned_words.append(lemmatizer.lemmatize(w))

    return " ".join(cleaned_words)


# Feature Extraction

def extract_text_features(df: pd.DataFrame, text_column: str) -> pd.DataFrame:

    df = df.copy()
    text_series = df[text_column].astype(str)


    # Structural Features

    df["char_length"] = text_series.str.len()
    df["word_count"] = text_series.str.split().apply(len)

    df["avg_word_length"] = np.where(
        df["word_count"] > 0,
        df["char_length"] / df["word_count"],
        0.0
    )


    # Stylistic Features

    df["caps_ratio"] = text_series.apply(
        lambda x: sum(1 for c in x if c.isupper()) / max(len(x), 1)
    )

    df["punctuation_density"] = text_series.apply(
        lambda x: len(PUNCT_PATTERN.findall(x)) / max(len(x), 1)
    )

    df["exclamation_count"] = text_series.str.count("!")


    # Behavioural Features (normalised signals)

    df["length_zscore"] = zscore(df["char_length"])
    df["word_count_zscore"] = zscore(df["word_count"])


    # Risk-oriented Heuristics

    df["short_text_flag"] = (df["char_length"] < 30).astype(int)
    df["high_caps_flag"] = (df["caps_ratio"] > 0.3).astype(int)
    df["excess_punctuation_flag"] = (df["punctuation_density"] > 0.1).astype(int)

    return df


# Main Pipeline

def preprocess_text_column(
    df: pd.DataFrame,
    text_column: str = "review_text",
    min_length: int = 20,
    remove_duplicates: bool = True,
    filter_non_english: bool = False,
    verbose: bool = True
) -> pd.DataFrame:

    df = df.copy()

    if verbose:
        print("Starting preprocessing pipeline...")


    # Validation

    if text_column not in df.columns:
        raise ValueError(f"Missing required column: {text_column}")

    df[text_column] = df[text_column].astype(str)
    df = df.dropna(subset=[text_column])


    # Deduplication

    if remove_duplicates:
        before = len(df)
        df = df.drop_duplicates(subset=[text_column])

        if verbose:
            print(f"Duplicates removed: {before - len(df)}")


    # Length Filter

    df = df[df[text_column].str.len() > min_length]


    # Language Filter

    if filter_non_english:
        df = df[df[text_column].apply(is_english)]


    # Feature Extraction (raw text)

    df = extract_text_features(df, text_column)


    # Clean Text Generation

    df["clean_text"] = df[text_column].apply(clean_text)

    # Remove only unusable rows
    df = df[df["clean_text"].str.len() > 0]


    # Numeric Safety Handling

    df.replace([np.inf, -np.inf], np.nan, inplace=True)

    numeric_cols = df.select_dtypes(include=[np.number]).columns

    # Median imputation (robust to outliers)
    df[numeric_cols] = df[numeric_cols].fillna(df[numeric_cols].median())


    # Final Sanity Check

    if verbose:
        print(f"Preprocessing complete. Final shape: {df.shape}")

    return df

In [9]:
# src/features.py

import numpy as np
import pandas as pd
from typing import Tuple, List

from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
from sentence_transformers import SentenceTransformer


# Model Initialization

sentiment_model = SentimentIntensityAnalyzer()
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")


# Sentiment Features

def compute_sentiment(texts: List[str]) -> np.ndarray:

    if not texts:
        return np.array([])

    return np.array([
        sentiment_model.polarity_scores(str(t))["compound"]
        for t in texts
    ], dtype=np.float32)


def add_sentiment_features(df: pd.DataFrame) -> pd.DataFrame:

    df = df.copy()

    if "clean_text" not in df.columns:
        raise ValueError("Missing required column: clean_text")

    df["sentiment"] = compute_sentiment(df["clean_text"].tolist())

    df["rating_norm"] = df["rating"] / 5.0

    df["sentiment_rating_gap"] = np.abs(
        df["sentiment"] - df["rating_norm"]
    )

    return df


# Embeddings

def generate_embeddings(texts: List[str], batch_size: int = 64) -> np.ndarray:

    if not texts:
        return np.zeros((0, 384), dtype=np.float32)

    embeddings = embedding_model.encode(
        texts,
        batch_size=batch_size,
        show_progress_bar=False,
        convert_to_numpy=True
    )

    embeddings = np.asarray(embeddings, dtype=np.float32)

    # L2 normalization (stabilises clustering + anomaly detection)
    norms = np.linalg.norm(embeddings, axis=1, keepdims=True)
    norms = np.clip(norms, 1e-10, None)

    embeddings = embeddings / norms

    return embeddings


def add_embeddings(df: pd.DataFrame) -> pd.DataFrame:

    df = df.copy()

    if "clean_text" not in df.columns:
        raise ValueError("Missing required column: clean_text")

    texts = df["clean_text"].fillna("").tolist()

    embeddings = generate_embeddings(texts)

    if len(embeddings) != len(df):
        raise ValueError("Embedding-row mismatch detected")

    df["embedding"] = list(embeddings)
    df["embedding_norm"] = np.linalg.norm(embeddings, axis=1)

    return df


# Derived Features

def add_derived_features(df: pd.DataFrame) -> pd.DataFrame:

    df = df.copy()

    required_cols = [
        "char_length",
        "word_count"
    ]

    for c in required_cols:
        if c not in df.columns:
            raise ValueError(f"Missing required feature column: {c}")

    df["word_density"] = df["word_count"] / (df["char_length"] + 1e-6)

    # Important:
    # I don't recompute z-scores here if already computed in preprocessing.
    # Will only create fallback if missing.

    if "length_zscore" not in df.columns:
        df["length_zscore"] = (
            (df["char_length"] - df["char_length"].mean()) /
            (df["char_length"].std() + 1e-6)
        )

    if "word_count_zscore" not in df.columns:
        df["word_count_zscore"] = (
            (df["word_count"] - df["word_count"].mean()) /
            (df["word_count"].std() + 1e-6)
        )

    return df


# Feature Matrix Builder

def build_feature_matrix(df: pd.DataFrame) -> Tuple[np.ndarray, List[str]]:

    feature_cols = [
        # Sentiment features
        "sentiment",
        "rating_norm",
        "sentiment_rating_gap",

        # Structural features
        "char_length",
        "word_count",
        "avg_word_length",

        # Stylistic features
        "caps_ratio",
        "punctuation_density",
        "exclamation_count",

        # Behavioral features
        "length_zscore",
        "word_count_zscore",
        "word_density",

        # Embedding metadata
        "embedding_norm"
    ]

    feature_cols = [c for c in feature_cols if c in df.columns]

    X_structured = df[feature_cols].to_numpy(dtype=np.float32)

    if "embedding" not in df.columns:
        raise ValueError("Missing embeddings for feature matrix construction")

    embeddings = np.stack(df["embedding"].values).astype(np.float32)

    if X_structured.shape[0] != embeddings.shape[0]:
        raise ValueError("Feature mismatch between structured data and embeddings")

    X = np.hstack([X_structured, embeddings])

    return X, feature_cols


# Full Pipeline

def build_features(df: pd.DataFrame) -> Tuple[pd.DataFrame, np.ndarray, List[str]]:

    df = df.copy()

    df = add_sentiment_features(df)
    df = add_embeddings(df)
    df = add_derived_features(df)

    X, feature_cols = build_feature_matrix(df)

    return df, X, feature_cols

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [10]:
# src/models.py

import numpy as np
import pandas as pd
from typing import Dict, Tuple, List

from sklearn.ensemble import IsolationForest
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score


# Feature Splitting

def split_features(X: np.ndarray, n_structured: int) -> Tuple[np.ndarray, np.ndarray]:

    if X.shape[1] <= n_structured:
        raise ValueError("Invalid feature split: structured dimension exceeds X shape.")

    X_structured = X[:, :n_structured]
    X_embeddings = X[:, n_structured:]

    return X_structured, X_embeddings


# Safe Scaling

def scale_features(X: np.ndarray) -> Tuple[StandardScaler, np.ndarray]:

    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    # Hard safety for NaN/inf leakage
    X_scaled = np.nan_to_num(X_scaled)

    return scaler, X_scaled


# Anomaly Detection

def train_isolation_forest(
    X: np.ndarray,
    contamination: float = 0.05
):

    model = IsolationForest(
        n_estimators=300,
        contamination=contamination,
        random_state=42,
        n_jobs=-1
    )

    model.fit(X)

    anomaly_score = model.decision_function(X)
    preds = model.predict(X)

    fake_flag = (preds == -1).astype(int)

    return model, anomaly_score, fake_flag


# Clustering

def train_kmeans(X: np.ndarray, n_clusters: int = 6):

    model = KMeans(
        n_clusters=n_clusters,
        random_state=42,
        n_init=10
    )

    clusters = model.fit_predict(X)

    # Distance to assigned centroid
    distances = np.linalg.norm(
        X - model.cluster_centers_[clusters],
        axis=1
    )

    sil_score = None
    if len(np.unique(clusters)) > 1:
        try:
            sil_score = silhouette_score(X, clusters)
        except Exception:
            sil_score = None

    return model, clusters, distances, sil_score


# Risk Feature Engineering

def build_risk_features(df: pd.DataFrame) -> np.ndarray:

    def safe(name: str, default: float = 0.0) -> np.ndarray:

        if name not in df.columns:
            return np.full(len(df), default)

        col = df[name]

        if col.isna().all():
            return np.full(len(df), default)

        return col.fillna(default).to_numpy()

    features = np.vstack([
        safe("anomaly_score"),
        safe("sentiment_rating_gap"),
        safe("cluster_distance"),
        safe("length_zscore"),
        safe("caps_ratio")
    ]).T

    return np.nan_to_num(features)


# Risk Scoring Engine

def compute_risk_score(df: pd.DataFrame) -> np.ndarray:


    # Anomaly Normalisation

    anomaly = df.get("anomaly_score", pd.Series(np.zeros(len(df))))

    anomaly_min = anomaly.min()
    anomaly_max = anomaly.max()

    anomaly_norm = 1 - (
        (anomaly - anomaly_min) /
        (anomaly_max - anomaly_min + 1e-6)
    )


    # Cluster Distance Normalisation

    dist = df.get("cluster_distance", pd.Series(np.zeros(len(df))))

    dist_min = dist.min()
    dist_max = dist.max()

    dist_norm = (
        (dist - dist_min) /
        (dist_max - dist_min + 1e-6)
    )


    # Sentiment Gap

    sentiment_gap = df.get(
        "sentiment_rating_gap",
        pd.Series(np.zeros(len(df)))
    )


    # Length Signal

    length_z = np.abs(df.get("length_zscore", pd.Series(np.zeros(len(df)))))


    # Weighted Fusion Score

    risk = (
        0.4 * anomaly_norm +
        0.3 * sentiment_gap +
        0.2 * dist_norm +
        0.1 * length_z
    )

    # Final normalization
    risk_min = risk.min()
    risk_max = risk.max()

    risk = (risk - risk_min) / (risk_max - risk_min + 1e-6)

    return np.nan_to_num(risk)


# Main Model Pipeline

def run_models(
    df: pd.DataFrame,
    X: np.ndarray,
    feature_cols: List[str],
    n_clusters: int = 6,
    contamination: float = 0.05
) -> Dict:

    df = df.copy()


    # Split Feature Space

    n_structured = len(feature_cols)

    X_structured, X_embeddings = split_features(X, n_structured)


    # Independent Scaling

    scaler_struct, Xs = scale_features(X_structured)
    scaler_embed, Xe = scale_features(X_embeddings)


    # Feature Fusion Layer

    X_final = np.hstack([
        Xs,
        Xe * 0.7  # Embedding down-weighting (stabilises clustering)
    ])

    X_final = np.nan_to_num(X_final)


    # Anomaly Model

    iso_model, anomaly_score, fake_flag = train_isolation_forest(
        X_final,
        contamination
    )

    df["anomaly_score"] = anomaly_score
    df["fake_review_flag"] = fake_flag


    # Clustering Model

    kmeans_model, clusters, distances, sil_score = train_kmeans(
        Xe,
        n_clusters
    )

    df["cluster"] = clusters
    df["cluster_distance"] = distances


    # Risk Engine

    df["risk_score"] = compute_risk_score(df)

    risk_features = build_risk_features(df)


    # Output Contract

    return {
        "df": df,
        "X_final": X_final,
        "risk_features": risk_features,

        "models": {
            "isolation_forest": iso_model,
            "kmeans": kmeans_model
        },

        "scalers": {
            "structured": scaler_struct,
            "embeddings": scaler_embed
        },

        "metrics": {
            "silhouette_score": sil_score
        }
    }

In [11]:
# src/calibration.py

import numpy as np
from typing import Tuple, Optional

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.calibration import CalibratedClassifierCV


# Train Calibration Model

def train_calibration_model(
    X: np.ndarray,
    y: np.ndarray,
    test_size: float = 0.2,
    random_state: int = 42
) -> Tuple[CalibratedClassifierCV, StandardScaler]:

    """
    Trains a calibrated probability model for fraud/risk scoring.

    Uses:
    - Logistic Regression (base model)
    - Platt scaling via CalibratedClassifierCV (true calibration layer)
    """


    # Safety Checks

    if len(np.unique(y)) < 2:
        raise ValueError("Calibration failed: only one class present in y.")

    if X.shape[0] != len(y):
        raise ValueError("X and y size mismatch.")


    # Train/test Split

    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=test_size,
        random_state=random_state,
        stratify=y
    )


    # Feature Scaling

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)

    
    # Important:
    # Calibration must be trained ONLY on training data 
    # to avoid probability leakage

    base_model = LogisticRegression(
        max_iter=1000,
        C=1.0,
        solver="lbfgs"
    )


    # Probability Calibration

    calibrated_model = CalibratedClassifierCV(
        estimator=base_model,
        method="sigmoid",   # Platt scaling (stable for medium datasets)
        cv=3
    )

    calibrated_model.fit(X_train_scaled, y_train)

    return calibrated_model, scaler


# Prediction

def calibrate_risk_model(
    model: CalibratedClassifierCV,
    scaler: StandardScaler,
    X: np.ndarray
) -> np.ndarray:

    """
    Returns calibrated probabilities (0-1).
    """

    X_scaled = scaler.transform(X)

    probs = model.predict_proba(X_scaled)[:, 1]

    return probs


# Full Pipeline

def run_calibration(
    X: np.ndarray,
    y: np.ndarray,
    return_model: bool = True
):
    """
    Full calibration pipeline:

    Returns:
    - Calibrated model
    - Scaler
    - Probability scores
    """

    model, scaler = train_calibration_model(X, y)

    probs = calibrate_risk_model(model, scaler, X)

    if return_model:
        return model, scaler, probs

    return probs

In [12]:
# src/explainability.py

import numpy as np
import pandas as pd
from collections import Counter
from typing import Dict


# Risk Banding

def get_risk_band(prob: float) -> str:

    if prob >= 0.8:
        return "high"
    elif prob >= 0.5:
        return "medium"
    else:
        return "low"


# Global Statistics Builder

def compute_global_stats(df: pd.DataFrame) -> Dict:

    """
    Computes stable thresholds for explanation logic.
    Must be computed ON training data in production.
    """

    required_cols = [
        "anomaly_score",
        "sentiment_rating_gap",
        "caps_ratio",
        "cluster_distance"
    ]

    stats = {}

    for col in required_cols:
        if col in df.columns:
            stats[col] = {
                "p10": df[col].quantile(0.1),
                "p90": df[col].quantile(0.9),
                "p95": df[col].quantile(0.95)
            }

    return stats


# Review-level Explanation

def explain_review(row: pd.Series, stats: Dict) -> dict:

    reasons = []


    # Anomaly Signal

    if "anomaly_score" in row and "anomaly_score" in stats:
        if row["anomaly_score"] < stats["anomaly_score"]["p10"]:
            reasons.append("Highly unusual behavioural pattern")


    # Sentiment Mismatch

    if "sentiment_rating_gap" in row and "sentiment_rating_gap" in stats:
        if row["sentiment_rating_gap"] > stats["sentiment_rating_gap"]["p90"]:
            reasons.append("Strong mismatch between sentiment and rating")


    # Length Signal

    if "length_zscore" in row:
        if abs(row["length_zscore"]) > 2:
            reasons.append("Unusual review length distribution")


    # Caps / Spam Signal

    if "caps_ratio" in row and "caps_ratio" in stats:
        if row["caps_ratio"] > stats["caps_ratio"]["p95"]:
            reasons.append("Excessive capitalization (spam-like pattern)")


    # Cluster Outlier

    if "cluster_distance" in row and "cluster_distance" in stats:
        if row["cluster_distance"] > stats["cluster_distance"]["p90"]:
            reasons.append("Semantic outlier within cluster")


    # Final Risk Score

    prob = float(row.get("fraud_probability", 0.0))
    risk_band = get_risk_band(prob)

    return {
        "risk_score": float(row.get("risk_score", 0.0)),
        "fraud_probability": prob,
        "risk_level": risk_band,
        "reasons": reasons if reasons else ["No strong anomaly signals detected"]
    }


# Batch Review Explanation

def generate_review_explanations(
    df: pd.DataFrame,
    top_n: int = 10,
    use_training_stats: bool = False,
    stats: Dict = None
) -> pd.DataFrame:

    df = df.copy()

    # If no external stats provided, compute from current dataset
    # However, in production, stats MUST come from training data
    if stats is None:
        stats = compute_global_stats(df)

    # Rank high-risk reviews
    top_df = df.sort_values("risk_score", ascending=False).head(top_n).copy()

    top_df["explanation"] = top_df.apply(
        lambda row: explain_review(row, stats),
        axis=1
    )

    return top_df[[
        "review_text",
        "risk_score",
        "fraud_probability",
        "explanation"
    ]]


# Cluster Explanation

def generate_cluster_explanations(
    df: pd.DataFrame,
    top_n_words: int = 10
) -> dict:

    cluster_summary = {}

    if "cluster" not in df.columns:
        raise ValueError("Missing required column: cluster")

    for cluster_id in sorted(df["cluster"].unique()):

        cluster_df = df[df["cluster"] == cluster_id]


        # Text Signal Extraction

        if "clean_text" in cluster_df.columns:
            text = " ".join(cluster_df["clean_text"].astype(str))
            words = text.split()

            # Filter noisy tokens
            words = [w for w in words if len(w) > 2]
            top_words = Counter(words).most_common(top_n_words)
        else:
            top_words = []

 
        # Metrics (safe access)

        def safe_mean(col):
            return float(cluster_df[col].mean()) if col in cluster_df else 0.0

        def safe_rate(condition):
            return float(condition.mean()) if len(condition) > 0 else 0.0

        avg_risk = safe_mean("risk_score")
        avg_sentiment = safe_mean("sentiment")

        fraud_rate = safe_mean("fake_review_flag") if "fake_review_flag" in cluster_df else 0.0

        high_risk_ratio = (
            safe_rate(cluster_df["fraud_probability"] > 0.8)
            if "fraud_probability" in cluster_df
            else 0.0
        )

        cluster_summary[int(cluster_id)] = {
            "size": int(len(cluster_df)),
            "top_words": top_words,
            "avg_risk": avg_risk,
            "avg_sentiment": avg_sentiment,
            "fraud_rate": fraud_rate,
            "high_risk_ratio": high_risk_ratio
        }

    return cluster_summary

In [13]:
# src/temporal_analysis.py

import pandas as pd
import numpy as np


# Internal Time Handling

def _prepare_time(df: pd.DataFrame) -> pd.DataFrame:

    df = df.copy()

    if "timestamp" not in df.columns:
        raise ValueError("Missing required column: timestamp")

    df["timestamp"] = pd.to_datetime(df["timestamp"], errors="coerce")

    # Remove invalid timestamps
    df = df.dropna(subset=["timestamp"])

    return df


# Review Volume (velocity)

def compute_review_velocity(df: pd.DataFrame, window: str = "7D") -> pd.DataFrame:

    df = _prepare_time(df)

    ts = (
        df.set_index("timestamp")
          .resample(window)
          .size()
          .rename("review_count")
          .reset_index()
    )

    # Smoothing for trend stability
    ts["review_count_smooth"] = ts["review_count"].rolling(
        window=3, min_periods=1
    ).mean()

    return ts


# Sentiment Drift Analysis

def compute_sentiment_drift(df: pd.DataFrame, window: str = "7D") -> pd.DataFrame:

    df = _prepare_time(df)

    if "sentiment" not in df.columns:
        raise ValueError("Missing required column: sentiment")

    ts = (
        df.set_index("timestamp")
          .resample(window)["sentiment"]
          .mean()
          .reset_index()
    )

    ts["sentiment_smooth"] = ts["sentiment"].rolling(
        window=3, min_periods=1
    ).mean()

    # Expanding baseline (cumulative expectation)
    ts["sentiment_baseline"] = ts["sentiment"].expanding().mean()

    ts["sentiment_drift"] = ts["sentiment"] - ts["sentiment_baseline"]

    return ts


# Fraud Spike Detection

def detect_fraud_spikes(df: pd.DataFrame, window: str = "7D") -> pd.DataFrame:

    df = _prepare_time(df)

    if "fake_review_flag" not in df.columns:
        raise ValueError("Missing required column: fake_review_flag")

    ts = (
        df.set_index("timestamp")
          .resample(window)["fake_review_flag"]
          .mean()
          .reset_index()
    )


    # Local Baseline (rolling)

    ts["baseline"] = ts["fake_review_flag"].rolling(
        window=3, min_periods=1
    ).mean()

    ts["deviation"] = ts["fake_review_flag"] - ts["baseline"]


    # Robust Z-score

    mean = ts["fake_review_flag"].mean()
    std = ts["fake_review_flag"].std()

    std = max(std, 1e-6)

    ts["zscore"] = (ts["fake_review_flag"] - mean) / std


    # Spike Flag (robust threshold)

    # Slightly more stable than strict z > 2 in sparse data
    ts["spike"] = ts["zscore"] > 2.0

    return ts


# Full Temporal Pipeline

def run_temporal_analysis(df: pd.DataFrame) -> dict:

    """
    Full temporal intelligence pipeline.

    Produces multi-resolution signals:
    - 1D: high sensitivity (micro spikes)
    - 7D: smoothed trend (macro behaviour)
    """

    return {

        # Review Velocity
        "review_velocity_7d": compute_review_velocity(df, "7D"),
        "review_velocity_1d": compute_review_velocity(df, "1D"),

        # Sentiment Drift
        "sentiment_drift_7d": compute_sentiment_drift(df, "7D"),
        "sentiment_drift_1d": compute_sentiment_drift(df, "1D"),

        # Fraud Spikes
        "fraud_spikes_7d": detect_fraud_spikes(df, "7D"),
        "fraud_spikes_1d": detect_fraud_spikes(df, "1D"),
    }

In [14]:
# src/evaluation.py

import numpy as np
import pandas as pd
from collections import Counter
from typing import Dict, Optional

from sklearn.metrics import (
    silhouette_score,
    roc_auc_score,
    average_precision_score
)


# Dataset Diagnostics

def compute_basic_metrics(df: pd.DataFrame) -> Dict:

    return {
        "total_reviews": int(len(df)),
        "fake_rate": float(df["fake_review_flag"].mean()) if "fake_review_flag" in df else 0.0,
        "avg_sentiment": float(df["sentiment"].mean()) if "sentiment" in df else 0.0,
        "avg_rating": float(df["rating"].mean()) if "rating" in df else 0.0,
        "avg_risk_score": float(df["risk_score"].mean()) if "risk_score" in df else 0.0
    }


def feature_distribution_summary(df: pd.DataFrame) -> Dict:

    def safe(col: str, fn=np.std):
        return float(fn(df[col])) if col in df and len(df[col]) > 0 else 0.0

    return {
        "sentiment_std": safe("sentiment", np.std),
        "rating_std": safe("rating", np.std),
        "risk_score_std": safe("risk_score", np.std),
        "sentiment_skew": float(df["sentiment"].skew()) if "sentiment" in df else 0.0
    }


# Correlation Analysis

def correlation_analysis(df: pd.DataFrame) -> Dict:

    cols = [c for c in [
        "sentiment",
        "rating",
        "risk_score",
        "anomaly_score"
    ] if c in df.columns]

    if len(cols) < 2:
        return {}

    return df[cols].corr(numeric_only=True).to_dict()


# Anomaly Evaluation

def evaluate_anomaly_detection(df: pd.DataFrame) -> Dict:

    if "fake_review_flag" not in df.columns or "anomaly_score" not in df.columns:
        return {}

    y_true = df["fake_review_flag"]
    y_score = df["anomaly_score"]

    # Ensure valid binary labels
    if y_true.nunique() < 2:
        return {
            "roc_auc": None,
            "pr_auc": None,
            "note": "Insufficient label variance"
        }

    try:
        roc = roc_auc_score(y_true, y_score)
    except Exception:
        roc = None

    try:
        pr_auc = average_precision_score(y_true, y_score)
    except Exception:
        pr_auc = None

    return {
        "roc_auc": float(roc) if roc is not None else None,
        "pr_auc": float(pr_auc) if pr_auc is not None else None
    }


# Clustering Evaluation

def evaluate_clustering(X: np.ndarray, labels: np.ndarray) -> Dict:

    results = {}

    # Silhouette requires > 1 cluster
    if len(np.unique(labels)) > 1:
        try:
            results["silhouette"] = float(silhouette_score(X, labels))
        except Exception:
            results["silhouette"] = None
    else:
        results["silhouette"] = None

    # Entropy stability
    counts = np.array(list(Counter(labels).values()), dtype=np.float64)

    if counts.sum() == 0:
        results["cluster_entropy"] = 0.0
        return results

    probs = counts / counts.sum()

    entropy = -np.sum(probs * np.log(probs + 1e-9))

    results["cluster_entropy"] = float(entropy)

    return results


# Cluster Interpretation

def get_cluster_keywords(df: pd.DataFrame, top_n: int = 10) -> Dict:

    if "cluster" not in df.columns or "clean_text" not in df.columns:
        return {}

    cluster_words = {}

    for c in sorted(df["cluster"].unique()):

        cluster_df = df[df["cluster"] == c]

        text = " ".join(cluster_df["clean_text"].astype(str))

        words = [w for w in text.split() if len(w) > 2]

        cluster_words[int(c)] = Counter(words).most_common(top_n)

    return cluster_words


# Product Intelligence

def compute_product_scores(df: pd.DataFrame) -> Optional[pd.DataFrame]:

    if "product_id" not in df.columns:
        return None

    agg_dict = {}

    for col in ["sentiment", "rating", "fake_review_flag", "risk_score"]:
        if col in df.columns:
            agg_dict[col] = "mean"

    agg_dict["review_id"] = "count" if "review_id" in df.columns else "size"

    product_scores = df.groupby("product_id").agg(agg_dict)

    product_scores = product_scores.rename(columns={
        "review_id": "review_count"
    })

    if "sentiment" in product_scores and "rating" in product_scores:

        product_scores["quality_score"] = (
            product_scores["sentiment"] * 0.4 +
            (product_scores["rating"] / 5.0) * 0.4 -
            product_scores.get("fake_review_flag", 0) * 0.2
        )

    return product_scores


# High Risk Extraction

def get_high_risk_reviews(df: pd.DataFrame, top_n: int = 10) -> pd.DataFrame:

    if "risk_score" not in df.columns:
        return pd.DataFrame()

    return df.sort_values("risk_score", ascending=False).head(top_n)


# Risk Validation

def evaluate_risk_score(df: pd.DataFrame) -> Dict:

    def safe_corr(a, b):
        if a in df and b in df:
            return float(df[a].corr(df[b]))
        return None

    return {
        "risk_vs_sentiment_corr": safe_corr("risk_score", "sentiment"),
        "risk_vs_rating_corr": safe_corr("risk_score", "rating"),
        "risk_distribution_std": float(df["risk_score"].std()) if "risk_score" in df else 0.0
    }


# Full Evaluation Pipeline

def run_evaluation(df: pd.DataFrame, X: np.ndarray = None) -> Dict:

    results = {}

    results["basic_metrics"] = compute_basic_metrics(df)
    results["distribution"] = feature_distribution_summary(df)

    results["correlation"] = correlation_analysis(df)

    results["anomaly_eval"] = evaluate_anomaly_detection(df)

    if X is not None and "cluster" in df.columns:
        results["clustering"] = evaluate_clustering(X, df["cluster"].values)

    results["cluster_keywords"] = get_cluster_keywords(df)

    results["product_scores"] = compute_product_scores(df)

    results["high_risk_reviews"] = get_high_risk_reviews(df)

    results["risk_validation"] = evaluate_risk_score(df)

    return results

In [15]:
# src/visualization.py

import os
import matplotlib.pyplot as plt
import pandas as pd


# Risk Distribution Plot

def plot_risk_distribution(df: pd.DataFrame, save_path: str) -> str:

    if "risk_score" not in df.columns:
        raise ValueError("Missing required column: risk_score")

    os.makedirs(save_path, exist_ok=True)

    plt.figure(figsize=(8, 5))

    plt.hist(df["risk_score"].dropna(), bins=50, alpha=0.75)

    plt.title("Risk Score Distribution")
    plt.xlabel("Risk Score")
    plt.ylabel("Frequency")

    output_file = os.path.join(save_path, "risk_distribution.png")

    plt.tight_layout()
    plt.savefig(output_file)
    plt.close()

    return output_file


# Cluster Distribution Plot

def plot_cluster_distribution(df: pd.DataFrame, save_path: str) -> str:

    if "cluster" not in df.columns:
        raise ValueError("Missing required column: cluster")

    os.makedirs(save_path, exist_ok=True)

    plt.figure(figsize=(8, 5))

    cluster_counts = df["cluster"].value_counts().sort_index()

    if cluster_counts.empty:
        raise ValueError("Cluster column is empty or invalid")

    cluster_counts.plot(kind="bar")

    plt.title("Cluster Distribution")
    plt.xlabel("Cluster ID")
    plt.ylabel("Number of Reviews")

    output_file = os.path.join(save_path, "cluster_distribution.png")

    plt.tight_layout()
    plt.savefig(output_file)
    plt.close()

    return output_file


# Risk vs Cluster View

def plot_risk_by_cluster(df: pd.DataFrame, save_path: str) -> str:

    if "cluster" not in df.columns or "risk_score" not in df.columns:
        raise ValueError("Missing required columns: cluster, risk_score")

    os.makedirs(save_path, exist_ok=True)

    plt.figure(figsize=(10, 6))

    df.groupby("cluster")["risk_score"].mean().plot(kind="bar")

    plt.title("Average Risk Score by Cluster")
    plt.xlabel("Cluster")
    plt.ylabel("Avg Risk Score")

    output_file = os.path.join(save_path, "risk_by_cluster.png")

    plt.tight_layout()
    plt.savefig(output_file)
    plt.close()

    return output_file

In [16]:
# src/pipeline.py

import pandas as pd
import numpy as np
import random

from src.data_loader import load_data
from src.preprocessing import preprocess_text_column
from src.features import build_features
from src.models import run_models
from src.evaluation import run_evaluation

from src.explainability import (
    generate_review_explanations,
    generate_cluster_explanations
)

from src.temporal_analysis import run_temporal_analysis
from src.calibration import run_calibration

from src.utils import setup_logger, Timer, save_json, save_model


# Reproducibility

def set_seed(seed: int = 42):
    np.random.seed(seed)
    random.seed(seed)


# Pipeline Orchestrator

def run_pipeline(
    reviews_path: str,
    metadata_path: str = None,
    n_clusters: int = 6,
    contamination: float = 0.05,
    verbose: bool = True,
    save_artifacts: bool = True,
    seed: int = 42
):

    set_seed(seed)

    logger = setup_logger()
    logger.info("Starting Product Intelligence Pipeline")


    # Data Loading

    with Timer("Data Loading"):
        df_reviews, df_meta = load_data(
            reviews_path,
            metadata_path,
            verbose=verbose
        )

    logger.info(f"Data loaded: {df_reviews.shape[0]} rows")
    logger.info(f"Columns detected: {df_reviews.columns.tolist()}")


    # Preprocessing

    with Timer("Preprocessing"):
        df = preprocess_text_column(
            df_reviews,
            text_column="review_text"
        )

    logger.info(f"Preprocessing complete: {df.shape[0]} rows")


    # Feature Engineering

    with Timer("Feature Engineering"):
        df, X, feature_cols = build_features(df)

    logger.info(f"Feature engineering complete: {len(feature_cols)} features")


    # Modeling

    with Timer("Model Training"):
        model_outputs = run_models(
            df=df,
            X=X,
            feature_cols=feature_cols,
            n_clusters=n_clusters,
            contamination=contamination
        )

    df = model_outputs["df"]

    logger.info("Model training complete")


    # Calibration Block

    with Timer("Calibration"):

        risk_features = model_outputs["risk_features"]

        if "fake_review_flag" not in df.columns:
            logger.warning("Missing fake_review_flag — skipping calibration")

            calibration_model = None
            calibration_scaler = None
            df["fraud_probability"] = 0.0

        else:
            y_dummy = df["fake_review_flag"].values
            unique_classes = np.unique(y_dummy)

            if len(unique_classes) < 2:
                logger.warning(
                    "Calibration skipped: only one class present in fake_review_flag"
                )

                calibration_model = None
                calibration_scaler = None
                df["fraud_probability"] = 0.0

            else:
                # Capture all outputs
                calibration_model, calibration_scaler, fraud_probs = run_calibration(
                    risk_features,
                    y_dummy
                )

                df["fraud_probability"] = fraud_probs

    logger.info("Calibration complete")

    # Temporal Analysis

    with Timer("Temporal Analysis"):
        temporal_results = run_temporal_analysis(df)

    logger.info("Temporal analysis complete")


    # Explainability

    with Timer("Explainability"):
        review_explanations = generate_review_explanations(df)
        cluster_explanations = generate_cluster_explanations(df)

    logger.info("Explainability complete")


    # Evaluation

    with Timer("Evaluation"):
        evaluation_results = run_evaluation(
            df=df,
            X=model_outputs["X_final"]
        )

    logger.info("Evaluation complete")

    # Visualisation

    from src.visualization import (
    plot_risk_distribution,
    plot_cluster_distribution
    )

    plot_risk_distribution(df, "outputs/charts")
    plot_cluster_distribution(df, "outputs/charts")


    # Artifact Persistence

    if save_artifacts:

        logger.info("Saving artifacts...")

        save_json(evaluation_results, "outputs/reports/evaluation.json")
        save_json(temporal_results, "outputs/reports/temporal_analysis.json")
        save_json(feature_cols, "outputs/reports/feature_columns.json")

        save_model(
            model_outputs["models"]["isolation_forest"],
            "models/isolation_forest.pkl"
        )

        save_model(
            model_outputs["models"]["kmeans"],
            "models/kmeans.pkl"
        )

        save_model(
            model_outputs["scalers"]["structured"],
            "models/scaler.pkl"
        )


    # Final Output Package

    results = {
        "data": df,

        "models": model_outputs["models"],

        "scalers": {
            **model_outputs["scalers"],
            "calibration": calibration_scaler
        },

        "calibration_model": calibration_model,

        "temporal_analysis": temporal_results,
        "review_explanations": review_explanations,
        "cluster_explanations": cluster_explanations,

        "evaluation": evaluation_results,
        "feature_columns": feature_cols
    }

    logger.info("Pipeline execution complete")

    return results

In [17]:
# run_pipeline.py

import yaml
import importlib
from pathlib import Path

from src.pipeline import run_pipeline
from src.utils import (
    save_dataframe,
    save_json,
    save_model,
    setup_logger
)


# Config Loading

def load_config(path: str = "config.yaml") -> dict:
    with open(path, "r") as f:
        return yaml.safe_load(f)


# Directory Safety Layer

def ensure_dirs(config: dict):
    """
    Ensures all required output directories exist.
    Prevents silent failures during artifact saving.
    """

    Path(config["output"]["processed_data_path"]).mkdir(parents=True, exist_ok=True)
    Path(config["output"]["model_path"]).mkdir(parents=True, exist_ok=True)
    Path(config["output"]["report_path"]).mkdir(parents=True, exist_ok=True)


# Main Execution Entry

def main():


    # Setup

    config = load_config()
    logger = setup_logger()

    logger.info("Starting Product Intelligence System")

    ensure_dirs(config)


    # Input Paths

    reviews_path = config["data"]["data_path"]
    metadata_path = config["data"].get("metadata_path")

    output_data_path = config["output"]["processed_data_path"]
    model_path = config["output"]["model_path"]
    report_path = config["output"]["report_path"]


    # Pipeline Execution

    logger.info("Running ML pipeline...")

    results = run_pipeline(
        reviews_path=reviews_path,
        metadata_path=metadata_path,
        n_clusters=config["model"]["n_clusters"],
        contamination=config["model"]["contamination"],
        verbose=True
    )

    df = results["data"]

    logger.info(f"Pipeline complete. Processed rows: {len(df)}")


    # Artifact Saving

    logger.info("Saving outputs...")


    # Dataset

    save_dataframe(
        df,
        f"{output_data_path}/processed_reviews.csv"
    )


    # Models

    models = results.get("models", {})

    if "isolation_forest" in models:
        save_model(
            models["isolation_forest"],
            f"{model_path}/isolation_forest.pkl"
        )

    if "kmeans" in models:
        save_model(
            models["kmeans"],
            f"{model_path}/kmeans.pkl"
        )

    if results.get("calibration_model") is not None:
        save_model(
            results["calibration_model"],
            f"{model_path}/calibration_model.pkl"
        )


    # Scalers (safe iteration)

    for name, scaler in results.get("scalers", {}).items():
        if scaler is not None:
            save_model(
                scaler,
                f"{model_path}/scaler_{name}.pkl"
            )


    # Reports (safe writes)

    if "evaluation" in results:
        save_json(
            results["evaluation"],
            f"{report_path}/evaluation.json"
        )

    if "cluster_explanations" in results:
        save_json(
            results["cluster_explanations"],
            f"{report_path}/cluster_explanations.json"
        )

    if "temporal_analysis" in results:
        save_json(
            results["temporal_analysis"],
            f"{report_path}/temporal_analysis.json"
        )


    # Tables

    if "review_explanations" in results:
        save_dataframe(
            results["review_explanations"],
            f"{report_path}/review_explanations.csv"
        )


    # Completion Log

    logger.info("All outputs saved successfully")
    logger.info("Pipeline finished successfully")


# Entry Point

if __name__ == "__main__":
    main()

[2026-04-26 19:25:12,597] [product_intelligence] INFO - Starting Product Intelligence System
[2026-04-26 19:25:12,606] [product_intelligence] INFO - Running ML pipeline...
[2026-04-26 19:25:12,609] [product_intelligence] INFO - Starting Product Intelligence Pipeline


Raw columns: ['id', 'dateAdded', 'dateUpdated', 'name', 'asins', 'brand', 'categories', 'primaryCategories', 'imageURLs', 'keys', 'manufacturer', 'manufacturerNumber', 'reviews.date', 'reviews.dateSeen', 'reviews.didPurchase', 'reviews.doRecommend', 'reviews.id', 'reviews.numHelpful', 'reviews.rating', 'reviews.sourceURLs', 'reviews.text', 'reviews.title', 'reviews.username', 'sourceURLs']


[2026-04-26 19:25:18,492] [product_intelligence] INFO - Data loaded: 18090 rows
[2026-04-26 19:25:18,508] [product_intelligence] INFO - Columns detected: ['id', 'dateAdded', 'dateUpdated', 'name', 'asins', 'brand', 'categories', 'primaryCategories', 'imageURLs', 'keys', 'manufacturer', 'manufacturerNumber', 'reviews.date', 'reviews.dateSeen', 'reviews.didPurchase', 'reviews.doRecommend', 'reviews.id', 'reviews.numHelpful', 'rating', 'reviews.sourceURLs', 'review_text', 'summary', 'reviews.username', 'sourceURLs', 'timestamp', 'review_id', 'product_id', 'user_id']


Final columns: ['id', 'dateAdded', 'dateUpdated', 'name', 'asins', 'brand', 'categories', 'primaryCategories', 'imageURLs', 'keys', 'manufacturer', 'manufacturerNumber', 'reviews.date', 'reviews.dateSeen', 'reviews.didPurchase', 'reviews.doRecommend', 'reviews.id', 'reviews.numHelpful', 'rating', 'reviews.sourceURLs', 'review_text', 'summary', 'reviews.username', 'sourceURLs', 'timestamp', 'review_id', 'product_id', 'user_id']
Final shape: (18090, 28)
Metadata not found — continuing without it.
Data Loading took 5.8651 seconds
Starting preprocessing pipeline...
Duplicates removed: 0


[2026-04-26 19:25:27,267] [product_intelligence] INFO - Preprocessing complete: 17042 rows


...

[2026-04-26 19:29:38,343] [product_intelligence] INFO - Explainability complete


Explainability took 1.0200 seconds


[2026-04-26 19:29:59,179] [product_intelligence] INFO - Evaluation complete


Evaluation took 20.8336 seconds


[2026-04-26 19:30:00,687] [product_intelligence] INFO - Saving artifacts...
[2026-04-26 19:30:01,098] [product_intelligence] INFO - Pipeline execution complete
[2026-04-26 19:30:01,125] [product_intelligence] INFO - Pipeline complete. Processed rows: 17042
[2026-04-26 19:30:01,129] [product_intelligence] INFO - Saving outputs...
[2026-04-26 19:31:22,794] [product_intelligence] INFO - All outputs saved successfully
[2026-04-26 19:31:22,795] [product_intelligence] INFO - Pipeline finished successfully


In [18]:
# app/dashboard.py

import streamlit as st
import pandas as pd
import numpy as np
import plotly.express as px


# Page Config

st.set_page_config(
    page_title="Product Intelligence System",
    layout="wide"
)

st.title("Product Intelligence & Trust System")
st.markdown(
    "Fraud detection • Product health • Explainability • Temporal monitoring"
)


# Data Loading

@st.cache_data(show_spinner=True)
def load_data():
    """
    Loads processed pipeline output safely.
    """

    path = "data/processed/processed_reviews.csv"

    try:
        df = pd.read_csv(path)
    except FileNotFoundError:
        st.error(f"Missing dataset at {path}. Run pipeline first.")
        return pd.DataFrame()

    return df


df = load_data()

if df.empty:
    st.stop()


# Safety Normalization

REQUIRED_COLS = [
    "risk_score",
    "sentiment",
    "cluster",
    "fake_review_flag"
]

for col in REQUIRED_COLS:
    if col not in df.columns:
        df[col] = 0


# Sidebar Filters

st.sidebar.header("Filters")

min_risk = st.sidebar.slider(
    "Minimum Risk Score",
    0.0, 1.0, 0.5
)

cluster_options = ["All"] + sorted(df["cluster"].dropna().unique().tolist())

selected_cluster = st.sidebar.selectbox(
    "Cluster",
    options=cluster_options
)


# Apply filters
filtered_df = df[df["risk_score"] >= min_risk]

if selected_cluster != "All":
    filtered_df = filtered_df[filtered_df["cluster"] == selected_cluster]


# KPI Section

st.subheader("System Overview")

def safe_mean(col):
    return round(filtered_df[col].mean(), 3) if len(filtered_df) > 0 else 0


col1, col2, col3, col4 = st.columns(4)

col1.metric("Total Reviews", len(filtered_df))
col2.metric("Avg Risk Score", safe_mean("risk_score"))
col3.metric("Fake Review Rate", safe_mean("fake_review_flag"))
col4.metric("Avg Sentiment", safe_mean("sentiment"))


# Risk Distribution

st.subheader("Risk Distribution")

fig = px.histogram(
    filtered_df,
    x="risk_score",
    nbins=50,
    title="Risk Score Distribution"
)

st.plotly_chart(fig, use_container_width=True)


# Cluster Distribution

st.subheader("Cluster Analysis")

cluster_counts = (
    filtered_df["cluster"]
    .value_counts()
    .reset_index()
)

cluster_counts.columns = ["cluster", "count"]

fig2 = px.bar(
    cluster_counts,
    x="cluster",
    y="count",
    title="Cluster Distribution"
)

st.plotly_chart(fig2, use_container_width=True)


# Sentiment vs Risk

st.subheader("Sentiment vs Risk")

fig3 = px.scatter(
    filtered_df,
    x="sentiment",
    y="risk_score",
    color="cluster",
    title="Sentiment vs Risk Score"
)

st.plotly_chart(fig3, use_container_width=True)


# Temporal View

st.subheader("Temporal Intelligence")

if "timestamp" in filtered_df.columns and len(filtered_df) > 0:

    temp = filtered_df.copy()

    temp["timestamp"] = pd.to_datetime(temp["timestamp"], errors="coerce")
    temp = temp.dropna(subset=["timestamp"])

    trend = (
        temp.set_index("timestamp")
        .resample("7D")["risk_score"]
        .mean()
        .reset_index()
    )

    fig4 = px.line(
        trend,
        x="timestamp",
        y="risk_score",
        title="Risk Trend Over Time"
    )

    st.plotly_chart(fig4, use_container_width=True)

else:
    st.info("No timestamp data available for temporal analysis.")


# High Risk Reviews

st.subheader("High Risk Reviews")

top_risk = filtered_df.sort_values(
    "risk_score",
    ascending=False
).head(10)

display_cols = [
    "review_text",
    "risk_score",
    "sentiment",
    "fake_review_flag",
    "cluster"
]

existing_cols = [c for c in display_cols if c in top_risk.columns]

st.dataframe(top_risk[existing_cols])


# Explainability Viewer

st.subheader("Explainability")

if "explanation" in filtered_df.columns:

    valid_idx = filtered_df.index[: min(100, len(filtered_df))]

    idx = st.selectbox(
        "Select Review Index",
        valid_idx
    )

    st.json(filtered_df.loc[idx, "explanation"])

else:
    st.info("Run pipeline with explainability enabled to view explanations.")


# Product Intelligence

if "product_id" in filtered_df.columns:

    st.subheader("Product Intelligence")

    product_scores = filtered_df.groupby("product_id").agg({
        "risk_score": "mean",
        "sentiment": "mean",
        "fake_review_flag": "mean"
    }).reset_index()

    fig5 = px.scatter(
        product_scores,
        x="sentiment",
        y="risk_score",
        size="fake_review_flag",
        title="Product Health Map"
    )

    st.plotly_chart(fig5, use_container_width=True)


# Raw Data Explorer

st.subheader("Raw Data Explorer")

st.dataframe(filtered_df.head(100))


st.success("Dashboard loaded successfully")

2026-04-26 19:31:25.941 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-26 19:31:25.946 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-26 19:31:26.989 
  command:

    streamlit run c:\Users\JoshuaMcCourt\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\ipykernel_launcher.py [ARGUMENTS]
2026-04-26 19:31:26.990 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-26 19:31:26.996 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-26 19:31:26.998 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-26 19:31:27.000 Thread 'MainThread': missing ScriptRunContext!

DeltaGenerator()

In [19]:
# config.yaml

project:
  name: "product-intelligence-system"
  version: "1.0.0"

seed: 42


# Data Layer

data:
  data_path: "data/raw/amazon_reviews.csv"
  metadata_path: "data/raw/product_metadata.csv"

  min_review_length: 20
  deduplicate: true
  filter_non_english: false


# Feature Layer

features:
  embedding_model: "all-MiniLM-L6-v2"
  batch_size: 64
  normalize_embeddings: true


# Model Layer

model:
  n_clusters: 6
  contamination: 0.05

  isolation_forest:
    n_estimators: 200
    max_samples: "auto"

  kmeans:
    n_init: 10
    max_iter: 300


# Calibration Layer

calibration:
  method: "logistic_regression"
  test_size: 0.2
  threshold: 0.5


# Temporal Intelligence

temporal:
  windows: ["1D", "7D", "30D"]
  default_window: "7D"

  smoothing_window: 3
  anomaly_zscore_threshold: 2.0


# Explainability

explainability:
  top_n_reviews: 10
  top_n_words: 10


# Evaluation

evaluation:
  enable_clustering_metrics: true
  enable_product_metrics: true


# Output Paths

output:
  processed_data_path: "data/processed/"
  model_path: "models/"
  scaler_path: "models/scalers/"
  report_path: "outputs/reports/"
  chart_path: "outputs/charts/"


# Visualization

visualization:
  theme: "plotly"
  max_points: 5000

SyntaxError: invalid syntax (1746341555.py, line 3)

In [ ]:
# requirements.txt

# Core Data Stack

pandas>=2.0.0
numpy>=1.24.0
scipy>=1.10.0


# NLP / Text Pipeline

nltk>=3.8.1
vaderSentiment>=3.3.2
sentence-transformers>=2.2.2
regex>=2023.0.0


# Machine Learning

scikit-learn>=1.3.0
joblib>=1.3.0


# Config + Pipeline Ops

pyyaml>=6.0
tqdm>=4.66.0


# Visualization

matplotlib>=3.7.0
seaborn>=0.12.0
plotly>=5.18.0


# Dashboard

streamlit>=1.28.0


# Model Interpretability

shap>=0.44.0


# MLOps Ready

mlflow>=2.8.0
rich>=13.0.0